# Treino Grupo C (QR-Lesion) — Kaggle T4

**Autor**: Massanori  
**Data**: 19/05/2026  
**Descrição**: Orquestra smoke + treino completo + Pearson sanity check do Grupo C (`QuantileRegressionLesionModule` com `qr_lesion_loss`, alpha=0.10, lambda_lesion=5.0) no S5. Contribuição original do TCC.

## Pré-requisitos

- Settings → Accelerator = **T4 x1** (ou P100)
- Add Input (DOIS datasets):
  1. **`tcc-mri-recons-varnet-brain-4x`** — reconstruções VarNet do S4
  2. **`tcc-mri-lesion-masks`** — máscaras `.pt` do S3 (precisa ser uploadado se ainda não estiver)

## Fluxo

1. Célula 2: clone do repo + instala dependencias
2. Célula 3: **diagnóstico** (GPU + DOIS datasets esperados)
3. Célula 4: **smoke 500 iters** (gate antes do treino full)
4. Célula 5: **treino completo 210k iters** (~6-12h GPU)
5. Célula 6: **Pearson sanity check global** (esperado ~similar ao B, não é a métrica da contribuição)
6. Célula 7: empacota outputs
7. Célula 8: publica como Kaggle dataset `tcc-mri-qr-lesion-checkpoints`

## Diferenças em relação ao Grupo B

- Loss: `qr_loss` → `qr_lesion_loss` com `lambda_lesion=5.0` (MVP)
- Adiciona `masks_dir` ao `ReconsSliceDataset`
- **Arquitetura idêntica ao B** (`QuantileRegressionLesionModule` é alias de `QuantileRegressionModule`) — controle experimental D4
- Mesma seed (42) e hiperparametros

## Nota sobre o Pearson check

A correlação global do C provavelmente será **similar à do B** (lesões são <5% dos pixels, dominando os pixels normais nivela a Pearson global). A diferença real do C aparece nas métricas **por região** que serão computadas no S5.7-S5.8 (`Coverage_lesion`, `IoU_uncertainty`, `ULAS`). O Pearson aqui é só sanity "não regrediu".

In [ ]:
# Célula 2 — Setup: clone repo + dependencias
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/KR0N0S7/tcc-mri-uncertainty.git'
REPO_DIR = Path('/kaggle/working/tcc-mri-uncertainty')

if REPO_DIR.is_dir():
    print(f'Repo ja existe em {REPO_DIR}; git pull...')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print('\nHEAD do repo:')
subprocess.run(['git', '-C', str(REPO_DIR), 'log', '-1', '--oneline'], check=False)

print('\nInstalando dependencias...')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '-r', str(REPO_DIR / 'requirements.txt'),
    'python-dotenv',
], check=True)
print('Setup OK')

In [ ]:
# Célula 3 — Diagnóstico OBRIGATÓRIO: GPU + DOIS datasets esperados.
import os
import subprocess
import sys
from pathlib import Path

RECONS_SLUG = 'tcc-mri-recons-varnet-brain-4x'
MASKS_SLUG = 'tcc-mri-lesion-masks'
EXPECTED_SUBS = {'train', 'val', 'cal', 'test'}


def kaggle_input_candidates(slug):
    """Retorna paths candidatos para um dataset Kaggle."""
    candidates = [Path('/kaggle/input') / slug]
    datasets_root = Path('/kaggle/input/datasets')
    if datasets_root.is_dir():
        for user_dir in datasets_root.iterdir():
            if user_dir.is_dir():
                candidates.append(user_dir / slug)
    return candidates


def find_recons_root(slug):
    for cand in kaggle_input_candidates(slug):
        if not cand.is_dir():
            continue
        subs = {p.name for p in cand.iterdir() if p.is_dir()}
        if EXPECTED_SUBS.issubset(subs):
            return cand
        sub_dirs = [p for p in cand.iterdir() if p.is_dir()]
        if len(sub_dirs) == 1 and (sub_dirs[0] / 'train').is_dir():
            return sub_dirs[0]
    raise FileNotFoundError(
        f'Recons dataset "{slug}" nao encontrado com train/val/cal/test.'
    )


def find_masks_root(slug, glob='*.pt', min_files=100):
    """Mascaras sao flat: encontra dir com >= min_files arquivos.pt."""
    for cand in kaggle_input_candidates(slug):
        if not cand.is_dir():
            continue
        n = len(list(cand.glob(glob)))
        if n >= min_files:
            return cand
        # Tenta aninhamento extra de 1 nivel
        sub_dirs = [p for p in cand.iterdir() if p.is_dir()]
        if len(sub_dirs) == 1:
            n_sub = len(list(sub_dirs[0].glob(glob)))
            if n_sub >= min_files:
                return sub_dirs[0]
    raise FileNotFoundError(
        f'Masks dataset "{slug}" nao encontrado (esperado >= {min_files} arquivos .pt).\n'
        f'Voce subiu o dataset de mascaras (S3) para o Kaggle? '
        f'Use o slug "{slug}" e adicione como Input deste notebook.'
    )


print('=' * 60)
print('DIAGNOSTICO')
print('=' * 60)

print('\n[1] Conteudo de /kaggle/input/:')
subprocess.run(['ls', '-la', '/kaggle/input/'], check=False)
if Path('/kaggle/input/datasets').is_dir():
    print('\n    Conteudo de /kaggle/input/datasets/:')
    subprocess.run(['ls', '-la', '/kaggle/input/datasets/'], check=False)

print(f'\n[2] Procurando recons dataset "{RECONS_SLUG}":')
recons_root = find_recons_root(RECONS_SLUG)
print(f'  -> RECONS_ROOT: {recons_root}')
for sub in ('train', 'val', 'cal', 'test'):
    p = recons_root / sub
    n_npz = len(list(p.glob('*.npz')))
    print(f'    {sub}: {n_npz} arquivos .npz')

print(f'\n[3] Procurando masks dataset "{MASKS_SLUG}":')
masks_root = find_masks_root(MASKS_SLUG)
n_masks = len(list(masks_root.glob('*.pt')))
print(f'  -> MASKS_ROOT: {masks_root}')
print(f'    {n_masks} arquivos .pt (esperado ~352)')

import torch
print(f'\n[4] CUDA available: {torch.cuda.is_available()}')
if not torch.cuda.is_available():
    raise RuntimeError('GPU nao habilitada. Settings -> Accelerator -> T4 x1.')
print(f'    GPU: {torch.cuda.get_device_name(0)}')
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'    VRAM: {vram_gb:.1f} GB')

os.environ['TCC_RECONS_DIR'] = str(recons_root)
os.environ['TCC_ANOTADOS_DIR'] = '/kaggle/working/_dummy_anotados'
os.environ['TCC_BRAIN_CSV'] = '/kaggle/working/_dummy_brain.csv'
Path('/kaggle/working/_dummy_anotados').mkdir(exist_ok=True)
Path('/kaggle/working/_dummy_brain.csv').touch()

print('\n[5] Config do projeto:')
subprocess.run([sys.executable, '-m', 'src.config'], check=True)

# Exporta para celulas seguintes
RECONS_ROOT = str(recons_root)
MASKS_ROOT = str(masks_root)

print('\n' + '=' * 60)
print('DIAGNOSTICO OK — seguro prosseguir')
print('=' * 60)

In [ ]:
# Célula 4 — Smoke: 500 iters do Grupo C (QR-Lesion, lambda=5.0).
import subprocess
import sys

smoke_run = '/kaggle/working/runs/group_c_smoke'

result = subprocess.run([
    sys.executable, 'scripts/smoke_train_qr_lesion.py',
    '--device', 'cuda',
    '--n-iters', '500',
    '--recons-dir', RECONS_ROOT,
    '--masks-dir', MASKS_ROOT,
    '--run-dir', smoke_run,
    '--lambda-lesion', '5.0',
])

assert result.returncode == 0, (
    'SMOKE FALHOU. Veja log acima. NAO prossiga para o treino completo.'
)
print('\nSMOKE OK — seguro rodar o treino completo na proxima celula.')

In [ ]:
# Célula 5 — Treino completo: 210000 iters do Grupo C.
# T4: ~6-12h. Resume automatico em run_dir/last.pt se a sessao cair.
import subprocess
import sys

full_run = '/kaggle/working/runs/group_c_full'

result = subprocess.run([
    sys.executable, 'scripts/train_qr_lesion.py',
    '--device', 'cuda',
    '--recons-dir', RECONS_ROOT,
    '--masks-dir', MASKS_ROOT,
    '--run-dir', full_run,
    '--num-workers', '2',
    '--lambda-lesion', '5.0',
])

assert result.returncode == 0, 'Treino falhou. Verifique log e last.pt.'
print('\nTreino completo concluido. Checkpoints em', full_run)

In [ ]:
# Célula 6 — Pearson sanity check.
# Como QuantileRegressionLesionModule = QuantileRegressionModule (alias),
# o mesmo eval_qr_pearson.py funciona aqui sem modificacao.
# Esperado: r similar ao do Grupo B (>= 0.85). Diferenca real do C
# aparece nas metricas por regiao no S5.7-S5.8.
import subprocess
import sys

result = subprocess.run([
    sys.executable, 'scripts/eval_qr_pearson.py',
    '--recons-dir', RECONS_ROOT,
    '--checkpoint', f'{full_run}/best.pt',
    '--split', 'val',
    '--device', 'cuda',
    '--threshold', '0.85',
])

assert result.returncode == 0, (
    'PEARSON FALHOU (r < 0.85). C regrediu em relacao ao paper base.'
)
print('\nPEARSON OK — r >= 0.85, comportamento global preservado.')

In [ ]:
# Célula 7 — Empacota outputs em pasta dedicada.
import shutil
from pathlib import Path

OUTPUT_DIR = Path('/kaggle/working/tcc-mri-qr-lesion-checkpoints')
OUTPUT_DIR.mkdir(exist_ok=True)

src_dir = Path(full_run)
for fname in ('last.pt', 'best.pt', 'metrics.csv', 'config_snapshot.json'):
    src = src_dir / fname
    if src.exists():
        shutil.copy2(src, OUTPUT_DIR / fname)
        size_mb = src.stat().st_size / 1e6
        print(f'  copiado: {fname} ({size_mb:.1f} MB)')
    else:
        print(f'  AUSENTE (pulando): {fname}')

print(f'\nOutput pronto em {OUTPUT_DIR}')

In [ ]:
# Célula 8 — Publica como Kaggle dataset via API.
import json
import subprocess
from pathlib import Path

KAGGLE_USER = 'massanorikishi'
DATASET_SLUG = 'tcc-mri-qr-lesion-checkpoints'
DATASET_TITLE = 'TCC MRI QR-Lesion Checkpoints (Grupo C, S5)'
OUTPUT_DIR = Path('/kaggle/working/tcc-mri-qr-lesion-checkpoints')

metadata = {
    'title': DATASET_TITLE,
    'id': f'{KAGGLE_USER}/{DATASET_SLUG}',
    'licenses': [{'name': 'CC0-1.0'}],
}
(OUTPUT_DIR / 'dataset-metadata.json').write_text(
    json.dumps(metadata, indent=2), encoding='utf-8'
)

print('Tentando criar dataset (primeira versao)...')
result = subprocess.run(
    ['kaggle', 'datasets', 'create', '-p', str(OUTPUT_DIR), '-r', 'zip'],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print('Create falhou, tentando version:')
    print(result.stderr)
    result = subprocess.run(
        ['kaggle', 'datasets', 'version', '-p', str(OUTPUT_DIR),
         '-m', 'Update from kaggle_train_groupC notebook', '-r', 'zip'],
        capture_output=True, text=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        print('ERRO:', result.stderr)
        raise RuntimeError('Falha ao criar/atualizar dataset.')

print(f'\nDataset: https://www.kaggle.com/datasets/{KAGGLE_USER}/{DATASET_SLUG}')

## Próximos passos (S5.7 — calibração conforme + métricas)

Com os 3 grupos treinados e checkpoints publicados (`tcc-mri-resm-checkpoints`, `tcc-mri-qr-checkpoints`, `tcc-mri-qr-lesion-checkpoints`):

### Calibração conforme (S5.7)

Notebook separado (`notebooks/kaggle_calibration_S5_7.ipynb`):

1. Carrega os 3 checkpoints (Grupos A, B, C) e roda inference no split `cal`
2. Para B e C: computa `lambda_cal` via nonconformity score (Romano et al., 2019, eq. 9-10):
   - Score_i = max(lower_i - target_i, target_i - upper_i)
   - lambda_cal = quantile-(1-alpha) com correcao (n+1)/n
   - Espera-se lambda_cal ~ 1.54 para o Grupo B (Giannakopoulos et al., 2026)
3. Calibra intervalos no split `test` aplicando lambda_cal

### Métricas globais e por região (S5.8)

Sobre o split `test` após calibração:

- **Coverage global**: % pixels com target ∈ [lower_cal, upper_cal] — esperado ~90% para todos
- **Coverage_lesion**: idem restrito às mascaras de lesao — onde C deve **superar** A e B
- **Mean Interval Width**: largura media — trade-off (narrower = better, sujeito a cobertura)
- **IoU_uncertainty**: alinhamento espacial entre regioes de alta uncertainty e regioes de alto erro
- **ULAS** (contribuição original): gradient alignment entre uncertainty e error dentro de lesoes

### Comparação estatística (S5.9)

- Friedman test + Nemenyi post-hoc (Demšar, 2006) para Coverage_lesion entre A/B/C
- Holm-Bonferroni correction para multiplas comparacoes
- BCa bootstrap para CIs em subgrupos pequenos
- Clopper-Pearson para Coverage_lesion (binomial exato)

Refs: Romano et al. (2019); Angelopoulos & Bates (2023); Demšar (2006); Giannakopoulos et al. (2026, seções IV-V).